# 🔧 Instructor Prep: Download & Quantize Models (FLUX-only edition)

**Run this notebook once, before the workshop — not by participants.**

It downloads T5-XXL, CLIP-L, and FLUX.1-schnell, quantizes T5-XXL and the FLUX transformer to 4-bit
NF4, and saves everything into a folder in **your own** Google Drive. Afterwards you share that
folder ("Anyone with the link → Viewer") and paste the link into the `SHARED_FOLDER_LINK` constant in
`00_complete_workshop.ipynb`. Participants then get a read-only shortcut to it in their own Drive.

**Storage — why this fits in 15GB:** T5-XXL and CLIP-L are each loaded **once** here and then handed
directly to `FluxPipeline` as its `text_encoder_2`/`text_encoder` — instead of letting the pipeline
download and save its *own* private copies (which is what pushed the shared folder past 15GB in an
earlier draft: T5-XXL alone is ~6GB, and saving it separately *and* embedded in the pipeline meant
paying for it twice). Expect roughly:

| Component | Size |
|---|---|
| T5-XXL (NF4) | ~6.2GB |
| FLUX.1-schnell transformer (NF4) | ~6.8GB |
| CLIP-L + VAE (fp16) | ~0.5GB |
| **Total** | **~13.5GB** |

That's what ends up **saved to Drive**, and it's the only thing that matters for the 15GB quota. It
is *not* what gets downloaded to get there, though — 4-bit quantization happens after the download, so
the source weights still arrive at whatever precision Hugging Face stores them in. Expect roughly
**~48GB of one-time download traffic** to this Colab VM's local disk (not Drive) to produce that
~13.5GB: T5 ~22GB (sourced from FLUX's own bf16 copy — the raw `google/t5-v1_1-xxl` checkpoint is
fp32 and nearly double that), the FLUX transformer ~24GB (bf16), CLIP-L ~1.7GB. **Nobody else pays
this cost** — participants, and every other section of the workshop, only ever touch the small
quantized result. The cells below clear the local download cache after each step so it doesn't build
up on the VM's disk (that cache is separate from — and doesn't affect — what's already saved to Drive).

**Runtime:** use a GPU runtime (T4 is fine — quantization happens during load, so the transformer
briefly needs to fit on whatever GPU you're running this on either way). Budget **30–45 minutes**
for this notebook depending on bandwidth.

In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title 2. Install packages
# Pinned exactly (not >=): an unbounded `-U transformers` picks up whatever is newest, and newer
# major releases have broken bitsandbytes save_pretrained() before (transformers 5.x's rewritten
# weight converter doesn't implement the save-direction path for quantized components yet, as of
# this writing — surfaces as a NotImplementedError inside core_model_loading.reverse_op). These are
# the versions the cluster setup was built and tested against.
!pip install -q "diffusers==0.36.0" "transformers==4.57.3" "accelerate==1.12.0" bitsandbytes sentencepiece peft huggingface_hub


In [ ]:
# @title 3. Imports, device check, and target paths
import os, shutil, gc
import torch
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device != "cuda":
    raise RuntimeError("Go to Runtime > Change runtime type and pick a GPU before running this notebook.")

# This becomes the folder you share with participants (see the last cell).
MODELS_DIR = Path('/content/drive/MyDrive/latent_vandalism_models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

T5_MODEL_PATH   = MODELS_DIR / "t5-v1_1-xxl-nf4"
CLIP_L_PATH     = MODELS_DIR / "clip-vit-large-patch14"
FLUX_MODEL_PATH = MODELS_DIR / "FLUX.1-schnell-nf4"

print(f"Target folder: {MODELS_DIR}")


In [ ]:
# @title 4. Hugging Face login
# black-forest-labs repos (FLUX.1-schnell included) require accepting their license on Hugging Face
# and logging in with a token, even though the weights themselves are openly licensed.
# Accept the license first: https://huggingface.co/black-forest-labs/FLUX.1-schnell
# Then create a token: https://huggingface.co/settings/tokens
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')  # a Colab secret named HF_TOKEN, if you've set one (🔑 icon in the left sidebar)
except Exception:
    pass

if not hf_token:
    from getpass import getpass
    hf_token = getpass('Paste your Hugging Face token (hf_...): ')  # input is hidden, not echoed to the cell output

login(token=hf_token)
print("✓ Logged in to Hugging Face")


## T5-XXL (shared text encoder, 4-bit NF4)

**Heads up on download size:** `bnb_4bit` quantization only shrinks what ends up on disk/in memory
*after* the download — the source weights still have to be fetched at whatever precision the repo
stores them in. The raw `google/t5-v1_1-xxl` checkpoint is fp32 (**~44.5GB**). Pulling T5 from
**FLUX's own repo** instead (`black-forest-labs/FLUX.1-schnell`, `text_encoder_2` subfolder, bf16)
gets the same weights for about **~22GB** — still a real one-time download (10–20 min depending on
bandwidth), just half the size. Once this cell finishes, everyone else — every participant, and every
other model here — reuses the ~6GB *quantized* result; nobody else pays this cost.

In [ ]:
from transformers import T5EncoderModel, T5Tokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

if T5_MODEL_PATH.exists():
    print(f"✓ {T5_MODEL_PATH} already exists — loading it (needed below for FLUX assembly)")
    try:
        tokenizer = T5Tokenizer.from_pretrained(T5_MODEL_PATH, local_files_only=True)
    except Exception as e:
        # A tokenizer saved by a different transformers version can end up missing files this one
        # expects (seen with T5's sentencepiece model file across a transformers 5.x -> 4.x pin
        # change) — that's a few MB, so just re-fetch the tokenizer rather than the whole model.
        print(f"⚠️  Saved tokenizer looks incompatible with this transformers version ({e});")
        print("   re-fetching just the tokenizer (a few MB, not the full model)...")
        tokenizer = T5Tokenizer.from_pretrained("black-forest-labs/FLUX.1-schnell", subfolder="tokenizer_2")
        tokenizer.save_pretrained(T5_MODEL_PATH)
    t5_model = T5EncoderModel.from_pretrained(T5_MODEL_PATH, local_files_only=True, device_map={"": 0})
else:
    print("Downloading T5 from FLUX's own repo (bf16, ~22GB) + quantizing to NF4...")
    tokenizer = T5Tokenizer.from_pretrained("black-forest-labs/FLUX.1-schnell", subfolder="tokenizer_2")
    t5_model = T5EncoderModel.from_pretrained(
        "black-forest-labs/FLUX.1-schnell", subfolder="text_encoder_2",
        quantization_config=bnb_config, device_map={"": 0},
    )
    tokenizer.save_pretrained(T5_MODEL_PATH)
    t5_model.save_pretrained(T5_MODEL_PATH)
    print(f"✓ Saved to {T5_MODEL_PATH}")


In [ ]:
# @title Free the ~22GB raw download from local disk (Colab VM, not Drive — the quantized copy is
# already safe on Drive above). Keeps local disk from filling up before the FLUX download below.
import shutil
from pathlib import Path
hf_cache = Path.home() / ".cache" / "huggingface"
if hf_cache.exists():
    freed = sum(f.stat().st_size for f in hf_cache.rglob('*') if f.is_file()) / 1024**3
    shutil.rmtree(hf_cache, ignore_errors=True)
    print(f"✓ Cleared {hf_cache} ({freed:.1f} GB freed on the Colab VM's local disk)")


## CLIP-L (small, kept fp16 — no quantization needed)

In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer

if CLIP_L_PATH.exists():
    print(f"✓ {CLIP_L_PATH} already exists — loading it (needed below for FLUX assembly)")
    clip_tokenizer = CLIPTokenizer.from_pretrained(CLIP_L_PATH, local_files_only=True)
    clip_model = CLIPTextModel.from_pretrained(CLIP_L_PATH, torch_dtype=torch.float16, local_files_only=True).to(device)
else:
    print("Downloading CLIP-L...")
    clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
    clip_model = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14", torch_dtype=torch.float16).to(device)
    clip_tokenizer.save_pretrained(CLIP_L_PATH)
    clip_model.save_pretrained(CLIP_L_PATH)
    print(f"✓ Saved to {CLIP_L_PATH}")


## FLUX.1-schnell (transformer only quantized — T5 + CLIP-L reused from above, not re-saved)

This is the step that actually avoids the duplicate-download problem: the transformer, VAE, and
scheduler are loaded as standalone components (not through `FluxPipeline.from_pretrained`, which
would try to fetch/allocate for *all* seven components at once), then assembled into a pipeline
object together with the T5/CLIP-L already loaded above — no `text_encoder_2` / `text_encoder`
download or re-save happens at all. The `shutil.rmtree` below is just a safety net in case that ever
changes.

**GPU memory note:** loading the transformer needs a large *transient* buffer on top of its own
final size (more so while quantizing fresh from bf16) — and T5 + CLIP-L, already loaded above, are
sitting on the GPU with no room left for that on a 16GB T4. The cell below frees them first (T5 can't
be moved with `.to()` once it's 4-bit quantized, so it's deleted and reloaded rather than moved),
loads the transformer on its own while the GPU has headroom, then brings T5/CLIP-L back — fast, from
the local copies already saved above, not re-downloaded — and assembles the pipeline by hand from the
finished pieces.

In [ ]:
from diffusers import FluxPipeline, FluxTransformer2DModel, AutoencoderKL, FlowMatchEulerDiscreteScheduler
from diffusers import BitsAndBytesConfig as DiffusersBnbConfig

def _free_text_encoders():
    global t5_model, clip_model
    del t5_model, clip_model
    # Belt-and-suspenders: Jupyter caches the result of any *other* cell that happened to end in a
    # bare expression referencing these objects (into Out[]/_/__/___), which would keep them pinned
    # in GPU memory despite the del above. Clearing that cache is what actually frees it.
    try:
        ip = get_ipython()
        if ip is not None:
            ip.user_ns["_"] = ip.user_ns["__"] = ip.user_ns["___"] = None
            if "Out" in ip.user_ns:
                ip.user_ns["Out"].clear()
    except NameError:
        pass
    gc.collect()
    torch.cuda.empty_cache()

def _reload_text_encoders():
    global t5_model, clip_model
    t5_model = T5EncoderModel.from_pretrained(T5_MODEL_PATH, local_files_only=True, device_map={"": 0})
    clip_model = CLIPTextModel.from_pretrained(CLIP_L_PATH, torch_dtype=torch.float16, local_files_only=True).to(device)

if FLUX_MODEL_PATH.exists():
    print(f"✓ {FLUX_MODEL_PATH} already exists, skipping")
else:
    print("Downloading + quantizing the FLUX transformer (large download, several minutes)...")
    _free_text_encoders()

    try:
        transformer_bnb_config = DiffusersBnbConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
        )
        transformer = FluxTransformer2DModel.from_pretrained(
            "black-forest-labs/FLUX.1-schnell", subfolder="transformer",
            quantization_config=transformer_bnb_config, torch_dtype=torch.float16,
        )
        vae = AutoencoderKL.from_pretrained("black-forest-labs/FLUX.1-schnell", subfolder="vae", torch_dtype=torch.float16)
        scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained("black-forest-labs/FLUX.1-schnell", subfolder="scheduler")
    finally:
        # ALWAYS restore T5/CLIP-L, even if the transformer download/load above raised — otherwise
        # a failed attempt (auth error, network blip, etc.) leaves them deleted for the rest of the
        # session, and every later cell looks like an unrelated "t5_model is not defined" bug.
        _reload_text_encoders()

    flux_pipe = FluxPipeline(
        scheduler=scheduler, vae=vae,
        text_encoder=clip_model, tokenizer=clip_tokenizer,
        text_encoder_2=t5_model, tokenizer_2=tokenizer,
        transformer=transformer,
    )
    flux_pipe.save_pretrained(FLUX_MODEL_PATH)
    for comp in ["text_encoder_2", "text_encoder"]:
        shutil.rmtree(FLUX_MODEL_PATH / comp, ignore_errors=True)
    del flux_pipe
    torch.cuda.empty_cache()
    print(f"✓ Saved to {FLUX_MODEL_PATH} (T5/CLIP-L excluded — they live at T5_MODEL_PATH / CLIP_L_PATH)")


In [ ]:
# @title Free the FLUX download from local disk too (same reasoning as after T5 above)
hf_cache = Path.home() / ".cache" / "huggingface"
if hf_cache.exists():
    freed = sum(f.stat().st_size for f in hf_cache.rglob('*') if f.is_file()) / 1024**3
    shutil.rmtree(hf_cache, ignore_errors=True)
    print(f"✓ Cleared {hf_cache} ({freed:.1f} GB freed on the Colab VM's local disk)")


## Done — check the footprint, then share the folder

In [ ]:
!du -sh "{MODELS_DIR}"/*
!echo "---"
!du -sh "{MODELS_DIR}"


### Share the folder with participants

1. In the Drive file browser (left sidebar in Colab, or drive.google.com), find **`latent_vandalism_models`**.
2. Right-click → **Share** → change access to **"Anyone with the link"** → **Viewer** → Copy link.
3. Paste that link as `SHARED_FOLDER_LINK` in the setup cell of `00_complete_workshop.ipynb` and
   commit/push, or just announce the link at the start of the workshop and have participants paste it
   in themselves.

Participants then run the "Mount Google Drive and link the shared models folder" cell in
`00_complete_workshop.ipynb`, which adds this folder as a **shortcut** in their Drive — shortcuts
don't count against *their* storage quota, only yours.